In [ ]:
import time
import pandas as pd
import numpy as np
import ast
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torchdiffeq import odeint_adjoint as odeint
from torch.optim.lr_scheduler import ReduceLROnPlateau
import warnings
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# 基本設定
warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# データ前処理
# ============================================================
def safe_parse_embedding(x):
    if pd.isna(x) or x == "":
        return None
    if isinstance(x, (list, np.ndarray)):
        return np.array(x, dtype=np.float32)
    if isinstance(x, str):
        try:
            s = re.sub(r'[\[\]\n]', '', x)
            s = s.replace(',', ' ')
            vals = np.array([float(v) for v in s.split() if v], dtype=np.float32)
            return vals if len(vals) > 0 else None
        except Exception:
            return None
    return None

def preprocess_data(file_path):
    print("1. データ読み込み開始...")
    df = pd.read_csv(file_path)
    
    print("2. 埋め込みベクトルを結合中...")
    combined_vectors = []
    valid_indices = []
    
    for i, row in df.iterrows():
        desc = safe_parse_embedding(row['description_embedding'])
        meta = safe_parse_embedding(row['metadata_embedding'])
        
        if desc is not None and meta is not None:
            combined_vectors.append(np.concatenate([desc, meta]))
            valid_indices.append(i)
        
        if i % 10000 == 0 and i > 0:
            print(f"  {i}件処理済み...")

    if not combined_vectors:
        return pd.DataFrame()

    df = df.iloc[valid_indices].copy()
    df['combined_vector'] = combined_vectors
    
    def parse_corp(x):
        try:
            return ast.literal_eval(x) if isinstance(x, str) else x
        except:
            return x
    
    df["corporation"] = df["corporation"].apply(parse_corp)
    df['year_month'] = pd.to_datetime(df['year_month'])
    df = df[(df['year_month'] >= '2010-01-01') & (df['year_month'] <= '2020-12-31')]
    
    print(f"✓ 前処理完了: {len(df)} 件")
    return df

# ============================================================
# GAT Encoder
# ============================================================
class SharedVGAEEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=4, concat=True, dropout=0.3)
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=2, concat=True, dropout=0.3)
        self.conv3 = GATConv(hidden_channels * 2, hidden_channels, heads=1, concat=False, dropout=0.2)
        
        self.conv_mu = GATConv(hidden_channels, out_channels, heads=1, concat=False)
        self.conv_logvar = GATConv(hidden_channels, out_channels, heads=1, concat=False)
        
        self.batch_norm1 = nn.BatchNorm1d(hidden_channels * 4)
        self.batch_norm2 = nn.BatchNorm1d(hidden_channels * 2)
        self.batch_norm3 = nn.BatchNorm1d(hidden_channels)

    def forward(self, x, edge_index):
        x = F.elu(self.batch_norm1(self.conv1(x, edge_index)))
        x = F.elu(self.batch_norm2(self.conv2(x, edge_index)))
        x = F.elu(self.batch_norm3(self.conv3(x, edge_index)))
        
        return self.conv_mu(x, edge_index), self.conv_logvar(x, edge_index)

# ============================================================
# Hamiltonian-ODE Components
# ============================================================
class PhysicalPotentialNet(nn.Module):
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.B = nn.Parameter(torch.randn(latent_dim + 1, hidden_dim // 2) * 2.0, requires_grad=False)
        
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, z):
        proj = torch.matmul(z, self.B)
        x = torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)
        return self.net(x)
    
    def compute_potential_grid(self, x_range, y_range, resolution=50, device='cpu'):
        """グリッド上でポテンシャルを計算"""
        x = torch.linspace(x_range[0], x_range[1], resolution)
        y = torch.linspace(y_range[0], y_range[1], resolution)
        X, Y = torch.meshgrid(x, y, indexing='ij')
        
        grid_points = torch.stack([X.flatten(), Y.flatten()], dim=1).to(device)
        with torch.no_grad():
            potentials = self.forward(grid_points)
        
        return X.cpu().numpy(), Y.cpu().numpy(), potentials.view(resolution, resolution).cpu().numpy()

class GradientODEFunc(nn.Module):
    def __init__(self, potential_net):
        super().__init__()
        self.potential_net = potential_net
        self.scale = nn.Parameter(torch.tensor(0.1))

    def forward(self, t, z):
        t_vec = torch.ones(z.size(0), 1).to(z.device) * t
        
        with torch.set_grad_enabled(True):
            z.requires_grad_(True)
            potential = self.potential_net(torch.cat([z, t_vec], dim=-1))
            grad = torch.autograd.grad(potential.sum(), z, create_graph=True)[0]
            
        return -self.scale * grad
    
    def compute_gradient_field(self, X, Y, device='cpu'):
        """グリッド上で勾配ベクトル場を計算"""
        grid_points = torch.tensor(
            np.stack([X.flatten(), Y.flatten()], axis=1), 
            dtype=torch.float32, device=device, requires_grad=True
        )
        
        phi = self.potential_net(grid_points)
        gradients = torch.autograd.grad(phi.sum(), grid_points, create_graph=False)[0]
        
        grad_x = gradients[:, 0].view(X.shape).cpu().numpy()
        grad_y = gradients[:, 1].view(X.shape).cpu().numpy()
        
        return grad_x, grad_y

class NeuralODEPredictor(nn.Module):
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.potential_net = PhysicalPotentialNet(latent_dim, hidden_dim)
        self.ode_func = GradientODEFunc(self.potential_net)

    def forward(self, z_current, delta_t=1.0):
        t_span = torch.tensor([0., delta_t], device=z_current.device)
        z_future = odeint(
            self.ode_func, z_current, t_span,
            method='dopri5', rtol=1e-3, atol=1e-3
        )[-1]
        return z_future

# ============================================================
# Base VGAE Model
# ============================================================
class UnifiedVGAE(nn.Module):
    def __init__(self, num_nodes, num_corps, input_dim, hidden_dim=128, latent_dim=2):
        super().__init__()
        self.num_nodes = num_nodes
        self.num_corps = num_corps
        self.latent_dim = latent_dim
        
        self.corp_embeddings = nn.Embedding(num_corps, input_dim)
        nn.init.normal_(self.corp_embeddings.weight, mean=0.0, std=0.1)
        
        self.encoder = SharedVGAEEncoder(input_dim, hidden_dim, latent_dim)
        self.temporal_predictor = NeuralODEPredictor(latent_dim, hidden_dim)
        
        self.link_predictor = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1)
        )

    def encode(self, x, edge_index):
        x_dynamic = x.clone()
        row, col = edge_index
        
        for c_i in range(self.num_corps):
            patent_indices = col[row == c_i]
            if len(patent_indices) > 0:
                mean_patent_feat = x[patent_indices].mean(dim=0)
                x_dynamic[c_i] = mean_patent_feat + self.corp_embeddings.weight[c_i]
            else:
                x_dynamic[c_i] = self.corp_embeddings.weight[c_i]
        
        mu, logvar = self.encoder(x_dynamic, edge_index)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z, edge_index):
        edge_features = torch.cat([z[edge_index[0]], z[edge_index[1]]], dim=-1)
        return torch.sigmoid(self.link_predictor(edge_features)).squeeze()

    def predict_future(self, z_current, delta_t=1.0):
        return self.temporal_predictor(z_current, delta_t)

# ============================================================
# Baseline Models
# ============================================================
class StaticVGAE(UnifiedVGAE):
    """Static baseline: no temporal evolution"""
    def predict_future(self, z_current, delta_t=1.0):
        return z_current

class LSTMVGAE(UnifiedVGAE):
    """LSTM baseline: discrete temporal modeling"""
    def __init__(self, num_nodes, num_corps, input_dim, hidden_dim=64, latent_dim=2):
        super().__init__(num_nodes, num_corps, input_dim, hidden_dim, latent_dim)
        self.lstm = nn.LSTM(latent_dim, hidden_dim, batch_first=True)
        self.fc_future = nn.Linear(hidden_dim, latent_dim)

    def predict_future(self, z_current, delta_t=1.0):
        z_in = z_current.unsqueeze(1) 
        out, _ = self.lstm(z_in)
        return self.fc_future(out.squeeze(1))

# ============================================================
# Loss Function
# ============================================================
def compute_loss(model, data_t, data_t1, z_history, num_corps, 
                historical_edges, alpha=0.5):
    device = data_t.x.device
    
    z_t, mu_t, logvar_t = model.encode(data_t.x, data_t.edge_index)
    
    pos_edges = data_t.edge_index
    pos_pred = model.decode(z_t, pos_edges)
    
    active_corps = torch.unique(pos_edges[0][pos_edges[0] < num_corps])
    active_patents = torch.unique(pos_edges[1][pos_edges[1] >= num_corps])
    
    neg_edges = []
    pos_set = set(tuple(e.tolist()) for e in pos_edges.t())
    
    for _ in range(min(500, pos_edges.size(1))):
        c = active_corps[torch.randint(len(active_corps), (1,))].item()
        p = active_patents[torch.randint(len(active_patents), (1,))].item()
        if (c, p) not in pos_set and (c, p) not in historical_edges:
            neg_edges.append([c, p])
    
    recon_loss = torch.tensor(0.0, device=device)
    if neg_edges:
        neg_edges_tensor = torch.tensor(neg_edges, device=device).t()
        neg_pred = model.decode(z_t, neg_edges_tensor)
        
        recon_loss = (
            F.binary_cross_entropy(pos_pred, torch.ones_like(pos_pred), reduction='mean') +
            F.binary_cross_entropy(neg_pred, torch.zeros_like(neg_pred), reduction='mean')
        )
    
    kl_loss = -0.5 * torch.mean(
        1 + logvar_t - mu_t.pow(2) - logvar_t.exp()
    )
    
    z_t1_pred = model.predict_future(z_history[-1])
    
    with torch.no_grad():
        _, mu_t1_actual, _ = model.encode(data_t1.x, data_t1.edge_index)
    
    future_loss = F.mse_loss(z_t1_pred[:num_corps], mu_t1_actual[:num_corps])
    
    total_loss = recon_loss + 0.01 * kl_loss + alpha * future_loss
    
    return total_loss, {
        'total': total_loss.item(),
        'recon': recon_loss.item(),
        'kl': kl_loss.item(),
        'future': future_loss.item()
    }

# ============================================================
# Evaluation Metrics
# ============================================================
def compute_ranking_metrics(model, data_current, data_future, num_corps, 
                            all_true_edges, k_values=[1, 3, 10, 50]):
    model.eval()
    pos_edges = data_future.edge_index
    
    if pos_edges.size(1) == 0:
        return None
    
    with torch.no_grad():
        z_current, _, _ = model.encode(data_current.x, data_current.edge_index)
        z_pred = model.predict_future(z_current)
    
    active_corps = torch.unique(pos_edges[0][pos_edges[0] < num_corps])
    active_patents = torch.unique(pos_edges[1][pos_edges[1] >= num_corps])
    
    if len(active_corps) == 0 or len(active_patents) == 0:
        return None
    
    reciprocal_ranks = []
    hits_at_k = {k: [] for k in k_values}
    
    num_eval = min(pos_edges.size(1), 200)
    
    for i in range(num_eval):
        src, dst = pos_edges[0, i].item(), pos_edges[1, i].item()
        
        filtered_negs = []
        for _ in range(200):
            neg_dst = active_patents[torch.randint(len(active_patents), (1,))].item()
            if neg_dst != dst and (src, neg_dst) not in all_true_edges:
                filtered_negs.append(neg_dst)
                if len(filtered_negs) >= 99:
                    break
        
        if len(filtered_negs) < 50:
            continue
        
        all_dsts = torch.tensor([dst] + filtered_negs, device=z_pred.device)
        src_repeated = torch.tensor([src] * len(all_dsts), device=z_pred.device)
        candidate_edges = torch.stack([src_repeated, all_dsts])
        
        with torch.no_grad():
            scores = model.decode(z_pred, candidate_edges).cpu().numpy()
        
        rank = (scores > scores[0]).sum() + 1
        reciprocal_ranks.append(1.0 / rank)
        
        for k in k_values:
            hits_at_k[k].append(1.0 if rank <= k else 0.0)
    
    if not reciprocal_ranks:
        return None
    
    metrics = {'mrr': np.mean(reciprocal_ranks)}
    for k in k_values:
        metrics[f'hits@{k}'] = np.mean(hits_at_k[k])
    
    return metrics

# ============================================================
# Training Loop (with history tracking)
# ============================================================
def train_model(model, graphs, num_corps, historical_edges, num_epochs=30):
    device = next(model.parameters()).device
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005, weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    
    years = sorted(graphs.keys())
    train_years = years[:int(len(years) * 0.7)]
    val_years = years[int(len(years) * 0.7):int(len(years) * 0.85)]
    
    all_true_edges = set()
    for year in years:
        edges = graphs[year].edge_index.t().tolist()
        all_true_edges.update([tuple(e) for e in edges])
    
    history = {'train_loss': [], 'val_mrr': [], 'epochs': []}
    best_val_mrr = 0.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        
        for i in range(len(train_years) - 1):
            year_t = train_years[i]
            year_t1 = train_years[i + 1]
            
            data_t = graphs[year_t].to(device)
            data_t1 = graphs[year_t1].to(device)
            
            z_history = []
            for j in range(max(0, i - 2), i + 1):
                with torch.no_grad():
                    data_hist = graphs[train_years[j]].to(device)
                    z_hist, _, _ = model.encode(data_hist.x, data_hist.edge_index)
                    z_history.append(z_hist)
            
            optimizer.zero_grad()
            loss, _ = compute_loss(
                model, data_t, data_t1, z_history, num_corps, historical_edges
            )
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / max(len(train_years) - 1, 1)
        
        model.eval()
        val_mrr_list = []
        
        for i in range(len(val_years) - 1):
            year_t = val_years[i]
            year_t1 = val_years[i + 1]
            
            data_t = graphs[year_t].to(device)
            data_t1 = graphs[year_t1].to(device)
            
            metrics = compute_ranking_metrics(
                model, data_t, data_t1, num_corps, all_true_edges
            )
            
            if metrics:
                val_mrr_list.append(metrics['mrr'])
        
        val_mrr = np.mean(val_mrr_list) if val_mrr_list else 0.0
        
        # Track history
        history['train_loss'].append(avg_loss)
        history['val_mrr'].append(val_mrr)
        history['epochs'].append(epoch + 1)
        
        if val_mrr > best_val_mrr:
            best_val_mrr = val_mrr
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        scheduler.step(val_mrr)
        print(f"Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | Val MRR: {val_mrr:.4f}")
    
    if best_model_state:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    
    return model, best_val_mrr, history

# ============================================================
# Graph Construction
# ============================================================
def build_global_graphs(df):
    all_corporations = sorted(list(set([c for corps in df['corporation'] for c in corps])))
    all_patents = sorted(df['patent_number'].unique().tolist())
    
    corp_to_idx = {corp: i for i, corp in enumerate(all_corporations)}
    patent_to_idx = {patent: i + len(all_corporations) for i, patent in enumerate(all_patents)}
    total_nodes = len(all_corporations) + len(all_patents)
    
    print(f"企業数: {len(all_corporations)}, 特許数: {len(all_patents)}")
    
    patent_features = {}
    for _, row in df.iterrows():
        patent_features[row['patent_number']] = row['combined_vector']

    global_graph_dict = {}
    year_groups = df.groupby(df['year_month'].dt.year)
    all_historical_edges = set()
    
    for year, group in year_groups:
        edges = []
        active_nodes = set()
        
        for _, row in group.iterrows():
            p_idx = patent_to_idx[row['patent_number']]
            active_nodes.add(p_idx)
            for corp in row['corporation']:
                c_idx = corp_to_idx[corp]
                edges.append([c_idx, p_idx])
                active_nodes.add(c_idx)
                all_historical_edges.add((c_idx, p_idx))
        
        if not edges:
            continue
        
        input_dim = len(next(iter(patent_features.values())))
        x = torch.zeros(total_nodes, input_dim)
        for p_num, p_idx in patent_to_idx.items():
            if p_num in patent_features:
                x[p_idx] = torch.tensor(patent_features[p_num])
        
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        active_mask = torch.zeros(total_nodes, dtype=torch.bool)
        active_mask[list(active_nodes)] = True
        
        global_graph_dict[year] = Data(
            x=x, 
            edge_index=edge_index, 
            active_mask=active_mask, 
            year=year,
            num_nodes=total_nodes
        )
        
    return (global_graph_dict, all_corporations, all_patents, 
            patent_to_idx, total_nodes, corp_to_idx, all_historical_edges, patent_features)

# ============================================================
# Visualization: Learning Curves
# ============================================================
def plot_learning_curves(all_histories):
    """Plot training loss and validation MRR for all models"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    colors = {'Static-VGAE': '#e74c3c', 'LSTM-VGAE': '#3498db', 'Hamiltonian-ODE': '#2ecc71'}
    
    # Training Loss
    for name, hist in all_histories.items():
        ax1.plot(hist['epochs'], hist['train_loss'], 
                label=name, linewidth=2, marker='o', markersize=4, 
                color=colors.get(name, 'gray'))
    
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Training Loss', fontsize=12)
    ax1.set_title('Training Loss Comparison', fontsize=14, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)
    
    # Validation MRR
    for name, hist in all_histories.items():
        ax2.plot(hist['epochs'], hist['val_mrr'], 
                label=name, linewidth=2, marker='s', markersize=4,
                color=colors.get(name, 'gray'))
    
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Validation MRR', fontsize=12)
    ax2.set_title('Validation MRR Comparison', fontsize=14, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# ============================================================
# Visualization: Corporate Trajectories
# ============================================================
def visualize_corporate_trajectories(model, graphs, num_corps, corp_to_idx, 
                                    all_corporations, device, 
                                    top_k=10, model_name="Model"):
    """Visualize continuous corporate trajectories across all years"""
    model.eval()
    years = sorted(graphs.keys())
    
    # Get embeddings for all years
    trajectories = {corp_idx: {'years': [], 'positions': []} 
                   for corp_idx in range(num_corps)}
    
    for year in years:
        data = graphs[year].to(device)
        with torch.no_grad():
            z, _, _ = model.encode(data.x, data.edge_index)
            z_corps = z[:num_corps].cpu().numpy()
        
        for corp_idx in range(num_corps):
            trajectories[corp_idx]['years'].append(year)
            trajectories[corp_idx]['positions'].append(z_corps[corp_idx])
    
    # Select top-k most active corporations
    corp_activity = {}
    for corp_idx in range(num_corps):
        total_movement = 0
        positions = trajectories[corp_idx]['positions']
        for i in range(len(positions) - 1):
            total_movement += np.linalg.norm(positions[i+1] - positions[i])
        corp_activity[corp_idx] = total_movement
    
    top_corps = sorted(corp_activity.items(), key=lambda x: x[1], reverse=True)[:top_k]
    top_corp_indices = [c[0] for c in top_corps]
    
    # Create visualization
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111)
    
    # Plot all patents as background
    all_z_patents = []
    for year in years:
        data = graphs[year].to(device)
        with torch.no_grad():
            z, _, _ = model.encode(data.x, data.edge_index)
            z_patents = z[num_corps:].cpu().numpy()
            all_z_patents.append(z_patents)
    
    all_patents_concat = np.vstack(all_z_patents)
    ax.scatter(all_patents_concat[:, 0], all_patents_concat[:, 1], 
              c='lightgray', s=10, alpha=0.2, label='Patents')
    
    # Plot corporate trajectories
    colormap = plt.cm.get_cmap('tab10')
    idx_to_corp = {v: k for k, v in corp_to_idx.items()}
    
    for i, corp_idx in enumerate(top_corp_indices):
        positions = np.array(trajectories[corp_idx]['positions'])
        years_list = trajectories[corp_idx]['years']
        
        color = colormap(i % 10)
        
        # Plot trajectory line
        ax.plot(positions[:, 0], positions[:, 1], 
               color=color, linewidth=2, alpha=0.7, zorder=10)
        
        # Plot yearly points
        ax.scatter(positions[:, 0], positions[:, 1], 
                  c=[color]*len(positions), s=80, 
                  edgecolors='black', linewidth=1.5, 
                  zorder=11, alpha=0.8)
        
        # Annotate start and end
        corp_name = idx_to_corp.get(corp_idx, f"Corp {corp_idx}")
        ax.annotate(f'{corp_name[:20]}\n({years_list[0]})', 
                   xy=positions[0], xytext=(10, 10),
                   textcoords='offset points', fontsize=8,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.5),
                   arrowprops=dict(arrowstyle='->', color='black', lw=1))
        
        ax.annotate(f'{years_list[-1]}', 
                   xy=positions[-1], xytext=(10, -10),
                   textcoords='offset points', fontsize=8,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.7))
    
    ax.set_title(f'Corporate Technology Trajectories ({model_name})', 
                fontsize=14, fontweight='bold')
    ax.set_xlabel('Technology Dimension 1', fontsize=12)
    ax.set_ylabel('Technology Dimension 2', fontsize=12)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ============================================================
# Visualization: Potential Field (for Hamiltonian-ODE only)
# ============================================================
def visualize_potential_field(model, graphs, year, device, num_corps):
    """Visualize potential field for Hamiltonian-ODE"""
    if not isinstance(model, UnifiedVGAE) or isinstance(model, (StaticVGAE, LSTMVGAE)):
        print(f"Potential field visualization only available for Hamiltonian-ODE")
        return
    
    model.eval()
    data = graphs[year].to(device)
    
    with torch.no_grad():
        z, _, _ = model.encode(data.x, data.edge_index)
        z_corps = z[:num_corps].cpu().numpy()
        z_patents = z[num_corps:].cpu().numpy()
    
    x = np.linspace(z_corps[:, 0].min()-1, z_corps[:, 0].max()+1, 100)
    y = np.linspace(z_corps[:, 1].min()-1, z_corps[:, 1].max()+1, 100)
    X, Y = np.meshgrid(x, y)
    
    grid = torch.tensor(np.stack([X.flatten(), Y.flatten()], axis=1), 
                       dtype=torch.float32).to(device)
    
    with torch.no_grad():
        phi = model.temporal_predictor.potential_net(grid)
    Z = phi.cpu().numpy().reshape(X.shape)
    
    fig, ax = plt.subplots(figsize=(12, 10))
    
    contour = ax.contourf(X, Y, Z, levels=20, cmap='RdYlBu', alpha=0.6)
    plt.colorbar(contour, label='Potential (Red=Active, Blue=Inactive)')
    
    ax.scatter(z_patents[:, 0], z_patents[:, 1], 
              c='gray', s=30, alpha=0.3, label='Patents')
    
    ax.scatter(z_corps[:, 0], z_corps[:, 1], 
              c='red', s=100, alpha=0.8, edgecolors='black', label='Corporations')
    
    # 勾配計算時も時刻情報を追加
    z_corps_tensor = torch.tensor(z_corps, device=device, requires_grad=True)
    t_vec_corps = torch.zeros(z_corps_tensor.size(0), 1).to(device)
    z_with_t = torch.cat([z_corps_tensor, t_vec_corps], dim=-1)
    
    phi_corps = model.temporal_predictor.potential_net(z_with_t) # z_with_tを使用
    grad = torch.autograd.grad(phi_corps.sum(), z_corps_tensor)[0].cpu().numpy()
    
    ax.quiver(z_corps[:, 0], z_corps[:, 1], -grad[:, 0], -grad[:, 1],
             color='black', alpha=0.7, scale=10, width=0.005,
             label='Strategic Direction')
    
    ax.set_title(f'Hamiltonian Potential Field ({year})', fontsize=14, fontweight='bold')
    ax.set_xlabel('Technology Dimension 1')
    ax.set_ylabel('Technology Dimension 2')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# ============================================================
# Comparative Experiment
# ============================================================
def run_comparative_experiment(graphs, num_corps, historical_edges, input_dim, 
                               total_nodes, corp_to_idx, all_corporations):
    models_config = {
        "Static-VGAE": StaticVGAE,
        "LSTM-VGAE": LSTMVGAE,
        "Hamiltonian-ODE": UnifiedVGAE
    }
    
    results = {}
    trained_models = {}
    all_histories = {}

    print(f"\n{'#'*60}")
    print(f"{'Starting Comparative Experiment':^60}")
    print(f"{'#'*60}\n")

    years = sorted(graphs.keys())
    all_true_edges = set()
    for y in years:
        edges = graphs[y].edge_index.t().tolist()
        all_true_edges.update([tuple(e) for e in edges])

    for name, model_cls in models_config.items():
        print(f"\n>>> [Training] {name}")
        
        m = model_cls(
            num_nodes=total_nodes, 
            num_corps=num_corps, 
            input_dim=input_dim, 
            hidden_dim=64, 
            latent_dim=2
        ).to(device)
        
        trained_m, _, history = train_model(
            m, graphs, num_corps, historical_edges, num_epochs=20
        )
        
        all_histories[name] = history
        
        test_year_t = years[-2]
        test_year_t1 = years[-1]
        
        print(f"--- Evaluating {name} on {test_year_t} -> {test_year_t1} ---")
        final_metrics = compute_ranking_metrics(
            trained_m, 
            graphs[test_year_t].to(device), 
            graphs[test_year_t1].to(device), 
            num_corps, 
            all_true_edges
        )
        
        results[name] = final_metrics
        trained_models[name] = trained_m

    # Results table
    print("\n" + "="*85)
    print(f"{'Table 1: Technology Trend Prediction Performance Comparison':^85}")
    print("="*85)
    print(f"{'Model Architecture':<25} | {'MRR':<10} | {'Hits@1':<10} | {'Hits@10':<10} | {'Hits@50':<10}")
    print("-" * 85)
    
    for name, metrics in results.items():
        if metrics:
            print(f"{name:<25} | {metrics['mrr']:.4f}    | {metrics['hits@1']:.4f}    | {metrics['hits@10']:.4f}    | {metrics['hits@50']:.4f}")
        else:
            print(f"{name:<25} | {'FAILED':<10} | {'N/A':<10} | {'N/A':<10} | {'N/A':<10}")
    print("="*85)
    
    # Visualizations
    print("\n[Visualization 1] Learning Curves...")
    plot_learning_curves(all_histories)
    
    print("\n[Visualization 2] Corporate Trajectories...")
    for name, model in trained_models.items():
        print(f"  - {name}")
        visualize_corporate_trajectories(
            model, graphs, num_corps, corp_to_idx, 
            all_corporations, device, top_k=10, model_name=name
        )
    
    print("\n[Visualization 3] Potential Field (Hamiltonian-ODE only)...")
    if "Hamiltonian-ODE" in trained_models:
        latest_year = max(graphs.keys())
        visualize_potential_field(
            trained_models["Hamiltonian-ODE"], 
            graphs, latest_year, device, num_corps
        )
    
    return trained_models, results, all_histories

# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    print("=" * 60)
    print("Hamiltonian-ODE vs Baselines Comparative Experiment")
    print("=" * 60)
    
    print("\n[Step 1] Data preprocessing...")
    df = preprocess_data('../dataset/topic_info3.csv')
    
    if len(df) == 0:
        print("❌ Empty dataset.")
        exit()
    
    print("\n[Step 2] Building dynamic graphs...")
    (graphs, corps, patents, p_map, total_n, c_map, 
     historical_edges, patent_features) = build_global_graphs(df)
    
    num_corps = len(corps)
    input_dim = len(df.iloc[0]['combined_vector'])
    
    print(f"✓ Total nodes: {total_n}")
    print(f"✓ Corporations: {num_corps}")
    print(f"✓ Years: {len(graphs)}")
    
    print("\n[Step 3] Running comparative experiment...")
    trained_models, final_results, all_histories = run_comparative_experiment(
        graphs, num_corps, historical_edges, input_dim, total_n, c_map, corps
    )
    
    print("\n✅ All experiments completed successfully.")

/home/nakamuraroi/.local/lib/python3.8/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/nakamuraroi/.local/lib/python3.8/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/nakamuraroi/.local/lib/python3.8/site-packages/torch_geometric/typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /home/nakamuraroi/.local/lib/python3.8/site-packages/torch_sparse/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "
/home/nakamuraroi/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook imp

Hamiltonian-ODE vs Baselines Comparative Experiment

[Step 1] Data preprocessing...
1. データ読み込み開始...
2. 埋め込みベクトルを結合中...
  10000件処理済み...
  20000件処理済み...
  30000件処理済み...
  40000件処理済み...
✓ 前処理完了: 19389 件

[Step 2] Building dynamic graphs...
企業数: 2450, 特許数: 19384
✓ Total nodes: 21834
✓ Corporations: 2450
✓ Years: 11

[Step 3] Running comparative experiment...

############################################################
              Starting Comparative Experiment               
############################################################


>>> [Training] Static-VGAE
Epoch 01 | Loss: 6.2983 | Val MRR: 0.0560
Epoch 02 | Loss: 4.1823 | Val MRR: 0.0549
Epoch 03 | Loss: 4.4437 | Val MRR: 0.0504
Epoch 04 | Loss: 4.9272 | Val MRR: 0.0528
Epoch 05 | Loss: 5.0656 | Val MRR: 0.0497
Epoch 06 | Loss: 5.2255 | Val MRR: 0.0620
Epoch 07 | Loss: 5.4820 | Val MRR: 0.0614
Epoch 08 | Loss: 5.3770 | Val MRR: 0.0611
Epoch 09 | Loss: 5.3886 | Val MRR: 0.0662
Epoch 10 | Loss: 5.0560 | Val MRR: 0.0634
Epoch 11 |

RuntimeError: mat1 and mat2 shapes cannot be multiplied (21834x3 and 2x32)

In [ ]:
import time
import pandas as pd
import numpy as np
import ast
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GATConv
from torchdiffeq import odeint_adjoint as odeint
from torch.optim.lr_scheduler import ReduceLROnPlateau
import warnings
import matplotlib.pyplot as plt

# 基本設定
warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ============================================================
# データ前処理
# ============================================================
def safe_parse_embedding(x):
    """埋め込みベクトルの文字列を安全にパースする"""
    if pd.isna(x) or x == "":
        return None
    if isinstance(x, (list, np.ndarray)):
        return np.array(x, dtype=np.float32)
    if isinstance(x, str):
        try:
            # 角括弧や改行を除去して数値化
            s = re.sub(r'[\[\]\n]', '', x)
            s = s.replace(',', ' ')
            vals = np.array([float(v) for v in s.split() if v], dtype=np.float32)
            return vals if len(vals) > 0 else None
        except Exception:
            return None
    return None

def preprocess_data(file_path):
    print("1. データ読み込み開始...")
    try:
        df = pd.read_csv(file_path)
    except FileNotFoundError:
        print(f"エラー: ファイル '{file_path}' が見つかりません。パスを確認してください。")
        return pd.DataFrame()

    print("2. 埋め込みベクトルを結合中...")
    combined_vectors = []
    valid_indices = []
    
    for i, row in df.iterrows():
        desc = safe_parse_embedding(row.get('description_embedding', ''))
        meta = safe_parse_embedding(row.get('metadata_embedding', ''))
        
        if desc is not None and meta is not None:
            # 次元の整合性チェック（例: 両方とも空でないか）
            if len(desc) > 0 and len(meta) > 0:
                combined_vectors.append(np.concatenate([desc, meta]))
                valid_indices.append(i)
        
        if i % 10000 == 0 and i > 0:
            print(f"  {i}件処理済み...")

    if not combined_vectors:
        print("警告: 有効なベクトルデータがありません。")
        return pd.DataFrame()

    df = df.iloc[valid_indices].copy()
    df['combined_vector'] = combined_vectors
    
    def parse_corp(x):
        try:
            # Pythonのリスト形式文字列を安全に評価
            return ast.literal_eval(x) if isinstance(x, str) else x
        except:
            # パースできない場合はリストにラップするか、そのまま返す
            return [x] if isinstance(x, str) else x
    
    if "corporation" in df.columns:
        df["corporation"] = df["corporation"].apply(parse_corp)
        # リストでないものをリスト化（エラー回避）
        df["corporation"] = df["corporation"].apply(lambda x: x if isinstance(x, list) else [])
    
    if "year_month" in df.columns:
        df['year_month'] = pd.to_datetime(df['year_month'], errors='coerce')
        df = df.dropna(subset=['year_month'])
        df = df[(df['year_month'] >= '2010-01-01') & (df['year_month'] <= '2020-12-31')]
    
    print(f"✓ 前処理完了: {len(df)} 件")
    return df

# ============================================================
# GAT Encoder
# ============================================================
class SharedVGAEEncoder(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        # Headsを調整（メモリ効率と学習安定性のため）
        self.conv1 = GATConv(in_channels, hidden_channels, heads=4, concat=True, dropout=0.3)
        # 入力次元は hidden * heads
        self.conv2 = GATConv(hidden_channels * 4, hidden_channels, heads=2, concat=True, dropout=0.3)
        self.conv3 = GATConv(hidden_channels * 2, hidden_channels, heads=1, concat=False, dropout=0.2)
        
        self.conv_mu = GATConv(hidden_channels, out_channels, heads=1, concat=False)
        self.conv_logvar = GATConv(hidden_channels, out_channels, heads=1, concat=False)
        
        self.batch_norm1 = nn.BatchNorm1d(hidden_channels * 4)
        self.batch_norm2 = nn.BatchNorm1d(hidden_channels * 2)
        self.batch_norm3 = nn.BatchNorm1d(hidden_channels)

    def forward(self, x, edge_index):
        x = F.elu(self.batch_norm1(self.conv1(x, edge_index)))
        x = F.elu(self.batch_norm2(self.conv2(x, edge_index)))
        x = F.elu(self.batch_norm3(self.conv3(x, edge_index)))
        
        return self.conv_mu(x, edge_index), self.conv_logvar(x, edge_index)

# ============================================================
# Hamiltonian-ODE Components
# ============================================================
class PhysicalPotentialNet(nn.Module):
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        # 時刻tを含めるため、入力次元は latent_dim + 1
        self.B = nn.Parameter(torch.randn(latent_dim + 1, hidden_dim // 2) * 2.0, requires_grad=False)
        
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.Tanh(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, 1)
        )

    def forward(self, z_with_t):
        # Random Fourier Featuresによる位置エンコーディング的な射影
        proj = torch.matmul(z_with_t, self.B)
        x = torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)
        return self.net(x)

class GradientODEFunc(nn.Module):
    def __init__(self, potential_net):
        super().__init__()
        self.potential_net = potential_net
        self.scale = nn.Parameter(torch.tensor(1.0)) # 初期スケール調整

    def forward(self, t, z):
        # zの勾配を計算するためにrequires_gradを有効化
        with torch.set_grad_enabled(True):
            z_in = z.requires_grad_(True)
            t_vec = torch.ones(z_in.size(0), 1).to(z_in.device) * t
            z_with_t = torch.cat([z_in, t_vec], dim=-1)
            
            potential = self.potential_net(z_with_t)
            # zに対する勾配のみを取得 (create_graph=Trueで高階微分に対応可能にする)
            grad = torch.autograd.grad(potential.sum(), z_in, create_graph=True)[0]
            
        return -self.scale * grad

class NeuralODEPredictor(nn.Module):
    def __init__(self, latent_dim, hidden_dim):
        super().__init__()
        self.potential_net = PhysicalPotentialNet(latent_dim, hidden_dim)
        self.ode_func = GradientODEFunc(self.potential_net)

    def forward(self, z_current, delta_t=1.0):
        t_span = torch.tensor([0., delta_t], device=z_current.device)
        z_future = odeint(
            self.ode_func, z_current, t_span,
            method='dopri5', rtol=1e-3, atol=1e-3
        )[-1]
        return z_future

# ============================================================
# Base VGAE Model
# ============================================================
class UnifiedVGAE(nn.Module):
    def __init__(self, num_nodes, num_corps, input_dim, hidden_dim=128, latent_dim=2):
        super().__init__()
        self.num_nodes = num_nodes
        self.num_corps = num_corps
        self.latent_dim = latent_dim
        
        # 企業の初期埋め込み（学習可能）
        self.corp_embeddings = nn.Embedding(num_corps, input_dim)
        nn.init.normal_(self.corp_embeddings.weight, mean=0.0, std=0.1)
        
        self.encoder = SharedVGAEEncoder(input_dim, hidden_dim, latent_dim)
        self.temporal_predictor = NeuralODEPredictor(latent_dim, hidden_dim)
        
        self.link_predictor = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_dim, 1)
        )

    def encode(self, x, edge_index):
        # 企業のEmbeddingを更新した特徴量を作成
        x_dynamic = x.clone()
        
        # ※GATがあるので、ここで平均を取る処理は必須ではないが、
        # 初期値として特許の平均を持たせるのは有効
        row, col = edge_index
        # 注意: edge_indexは無向グラフ化されている前提
        
        # GATエンコーダへ
        mu, logvar = self.encoder(x_dynamic, edge_index)
        z = self.reparameterize(mu, logvar)
        return z, mu, logvar

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu

    def decode(self, z, edge_index):
        # エッジの両端のノードの特徴を結合
        z_src = z[edge_index[0]]
        z_dst = z[edge_index[1]]
        edge_features = torch.cat([z_src, z_dst], dim=-1)
        return torch.sigmoid(self.link_predictor(edge_features)).squeeze()

    def predict_future(self, z_current, delta_t=1.0):
        return self.temporal_predictor(z_current, delta_t)

# ============================================================
# Baseline Models
# ============================================================
class StaticVGAE(UnifiedVGAE):
    """静的ベースライン: 時間変化なし"""
    def predict_future(self, z_current, delta_t=1.0):
        return z_current

class LSTMVGAE(UnifiedVGAE):
    """LSTMベースライン: 離散的な時間変化"""
    def __init__(self, num_nodes, num_corps, input_dim, hidden_dim=64, latent_dim=2):
        super().__init__(num_nodes, num_corps, input_dim, hidden_dim, latent_dim)
        # LSTM: input=latent, hidden=hidden
        self.lstm = nn.LSTM(latent_dim, hidden_dim, batch_first=True)
        self.fc_future = nn.Linear(hidden_dim, latent_dim)

    def predict_future(self, z_current, delta_t=1.0):
        # z_current: [num_nodes, latent_dim] -> input: [num_nodes, 1, latent_dim]
        z_in = z_current.unsqueeze(1) 
        out, _ = self.lstm(z_in)
        # out: [num_nodes, 1, hidden_dim] -> squeeze -> [num_nodes, hidden_dim]
        return self.fc_future(out.squeeze(1))

# ============================================================
# Loss Function
# ============================================================
def compute_loss(model, data_t, data_t1, z_history, num_corps, 
                historical_edges, alpha=0.5):
    device = data_t.x.device
    
    # 現時点の埋め込み
    z_t, mu_t, logvar_t = model.encode(data_t.x, data_t.edge_index)
    
    # 1. 再構成誤差 (Reconstruction Loss)
    pos_edges = data_t.edge_index
    
    # 負例サンプリング (Negative Sampling)
    # 高速化のため、エッジリストに含まれないペアをランダム生成
    num_neg = pos_edges.size(1)
    neg_edges = torch.randint(0, data_t.num_nodes, (2, num_neg), device=device)
    
    # 正例と負例の予測
    pos_pred = model.decode(z_t, pos_edges)
    neg_pred = model.decode(z_t, neg_edges)
    
    # 数値安定性のためのClamp
    pos_pred = torch.clamp(pos_pred, min=1e-6, max=1-1e-6)
    neg_pred = torch.clamp(neg_pred, min=1e-6, max=1-1e-6)
    
    recon_loss = -torch.mean(torch.log(pos_pred)) - torch.mean(torch.log(1 - neg_pred))
    
    # 2. KL発散 (KL Divergence)
    kl_loss = -0.5 * torch.mean(1 + logvar_t - mu_t.pow(2) - logvar_t.exp())
    
    # 3. 未来予測誤差 (Future Prediction Loss)
    # z_historyの最後(=z_tに相当)から未来を予測
    # LSTMの場合はhistory全体を使うのが理想だが、ここでは簡易的に現在の状態からの遷移とする
    z_t1_pred = model.predict_future(z_t)
    
    # 正解データの未来の潜在変数 (勾配を切る)
    with torch.no_grad():
        _, mu_t1_actual, _ = model.encode(data_t1.x, data_t1.edge_index)
    
    # 企業ノードのみのMSEを計算（特許はあまり移動しないと仮定、または企業戦略にフォーカス）
    future_loss = F.mse_loss(z_t1_pred[:num_corps], mu_t1_actual[:num_corps])
    
    total_loss = recon_loss + 0.01 * kl_loss + alpha * future_loss
    
    return total_loss, {
        'total': total_loss.item(),
        'recon': recon_loss.item(),
        'kl': kl_loss.item(),
        'future': future_loss.item()
    }

# ============================================================
# Evaluation Metrics
# ============================================================
def compute_ranking_metrics(model, data_current, data_future, num_corps, 
                            all_true_edges, k_values=[1, 3, 10, 50]):
    model.eval()
    pos_edges = data_future.edge_index
    
    if pos_edges.size(1) == 0:
        return None
    
    with torch.no_grad():
        z_current, _, _ = model.encode(data_current.x, data_current.edge_index)
        z_pred = model.predict_future(z_current)
    
    # 評価対象: 企業 -> 特許 のエッジのみ抽出
    # pos_edges[0]が企業、pos_edges[1]が特許であるペアを探す
    mask = (pos_edges[0] < num_corps) & (pos_edges[1] >= num_corps)
    eval_edges = pos_edges[:, mask]
    
    if eval_edges.size(1) == 0:
        return None
    
    active_patents = torch.unique(pos_edges[1][pos_edges[1] >= num_corps])
    if len(active_patents) < 2:
        return None
        
    reciprocal_ranks = []
    hits_at_k = {k: [] for k in k_values}
    
    # 計算時間短縮のためサンプリング
    num_eval = min(eval_edges.size(1), 200)
    indices = torch.randperm(eval_edges.size(1))[:num_eval]
    
    for idx in indices:
        src, dst = eval_edges[0, idx].item(), eval_edges[1, idx].item()
        
        # 負例（リンクしていない特許）をサンプリング
        neg_candidates = []
        while len(neg_candidates) < 99:
            neg = active_patents[torch.randint(len(active_patents), (1,))].item()
            if neg != dst and (src, neg) not in all_true_edges:
                neg_candidates.append(neg)
        
        # [正例, 負例...] のリストを作成
        targets = [dst] + neg_candidates
        src_repeated = torch.tensor([src] * len(targets), device=z_pred.device)
        dst_tensor = torch.tensor(targets, device=z_pred.device)
        
        candidate_edges = torch.stack([src_repeated, dst_tensor])
        
        with torch.no_grad():
            scores = model.decode(z_pred, candidate_edges).cpu().numpy()
        
        # 正例(scores[0])のランクを計算
        # スコアが高い順にソートしたとき、scores[0]が何番目か
        # (自分よりスコアが高いサンプルの数 + 1)
        rank = (scores > scores[0]).sum() + 1
        reciprocal_ranks.append(1.0 / rank)
        
        for k in k_values:
            hits_at_k[k].append(1.0 if rank <= k else 0.0)
    
    if not reciprocal_ranks:
        return None
    
    metrics = {'mrr': np.mean(reciprocal_ranks)}
    for k in k_values:
        metrics[f'hits@{k}'] = np.mean(hits_at_k[k])
    
    return metrics

# ============================================================
# Training Loop
# ============================================================
def train_model(model, graphs, num_corps, historical_edges, num_epochs=30):
    device = next(model.parameters()).device
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=5e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-5)
    
    years = sorted(graphs.keys())
    # データ分割
    split_idx = int(len(years) * 0.75)
    train_years = years[:split_idx]
    val_years = years[split_idx:]
    
    all_true_edges = set()
    for year in years:
        edges = graphs[year].edge_index.t().tolist()
        all_true_edges.update([tuple(e) for e in edges])
    
    history = {'train_loss': [], 'val_mrr': [], 'epochs': []}
    best_val_mrr = 0.0
    best_model_state = None
    
    print(f"Training on {len(train_years)} years, Validating on {len(val_years)} years")
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        steps = 0
        
        # 時系列ペアで学習 (t -> t+1)
        for i in range(len(train_years) - 1):
            year_t = train_years[i]
            year_t1 = train_years[i + 1]
            
            data_t = graphs[year_t].to(device)
            data_t1 = graphs[year_t1].to(device)
            
            optimizer.zero_grad()
            loss, _ = compute_loss(
                model, data_t, data_t1, [], num_corps, historical_edges
            )
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            steps += 1
        
        avg_loss = epoch_loss / max(steps, 1)
        
        # 検証
        model.eval()
        val_mrr_list = []
        
        for i in range(len(val_years) - 1):
            year_t = val_years[i]
            year_t1 = val_years[i + 1]
            
            data_t = graphs[year_t].to(device)
            data_t1 = graphs[year_t1].to(device)
            
            metrics = compute_ranking_metrics(
                model, data_t, data_t1, num_corps, all_true_edges
            )
            
            if metrics:
                val_mrr_list.append(metrics['mrr'])
        
        val_mrr = np.mean(val_mrr_list) if val_mrr_list else 0.0
        
        history['train_loss'].append(avg_loss)
        history['val_mrr'].append(val_mrr)
        history['epochs'].append(epoch + 1)
        
        if val_mrr >= best_val_mrr:
            best_val_mrr = val_mrr
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        
        scheduler.step(val_mrr)
        print(f"Epoch {epoch+1:02d} | Loss: {avg_loss:.4f} | Val MRR: {val_mrr:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
    
    if best_model_state:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
    
    return model, best_val_mrr, history

# ============================================================
# Graph Construction
# ============================================================
def build_global_graphs(df):
    # 企業と特許のリスト作成
    all_corporations = []
    for corps in df['corporation']:
        all_corporations.extend(corps)
    all_corporations = sorted(list(set(all_corporations)))
    
    all_patents = sorted(df['patent_number'].unique().tolist())
    
    corp_to_idx = {corp: i for i, corp in enumerate(all_corporations)}
    patent_to_idx = {patent: i + len(all_corporations) for i, patent in enumerate(all_patents)}
    total_nodes = len(all_corporations) + len(all_patents)
    
    print(f"企業数: {len(all_corporations)}, 特許数: {len(all_patents)}")
    
    # 特許特徴量マップ
    patent_features = {}
    for _, row in df.iterrows():
        patent_features[row['patent_number']] = row['combined_vector']
    
    # 特徴量の次元数
    if len(patent_features) > 0:
        input_dim = len(next(iter(patent_features.values())))
    else:
        input_dim = 10 # ダミー

    global_graph_dict = {}
    year_groups = df.groupby(df['year_month'].dt.year)
    all_historical_edges = set()
    
    for year, group in year_groups:
        edges = []
        active_nodes = set()
        
        for _, row in group.iterrows():
            if row['patent_number'] not in patent_to_idx: continue
            
            p_idx = patent_to_idx[row['patent_number']]
            active_nodes.add(p_idx)
            
            for corp in row['corporation']:
                if corp in corp_to_idx:
                    c_idx = corp_to_idx[corp]
                    # ★重要: GATのために双方向エッジを追加
                    edges.append([c_idx, p_idx]) # Corp -> Patent
                    edges.append([p_idx, c_idx]) # Patent -> Corp
                    
                    active_nodes.add(c_idx)
                    all_historical_edges.add((c_idx, p_idx))
        
        if not edges:
            continue
        
        # ノード特徴量の作成
        x = torch.zeros(total_nodes, input_dim)
        # 特許ノードにはBert/Word2Vec等の埋め込みを使用
        for p_num, p_idx in patent_to_idx.items():
            if p_num in patent_features:
                x[p_idx] = torch.tensor(patent_features[p_num])
        # 企業ノードは初期値0（モデル内でEmbedding加算または学習）
        
        edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
        
        global_graph_dict[year] = Data(
            x=x, 
            edge_index=edge_index, 
            year=year,
            num_nodes=total_nodes
        )
        
    return (global_graph_dict, all_corporations, all_patents, 
            patent_to_idx, total_nodes, corp_to_idx, all_historical_edges, input_dim)

# ============================================================
# Visualization
# ============================================================
def visualize_potential_field(model, graphs, year, device, num_corps):
    """Hamiltonian-ODEのポテンシャル場を可視化"""
    if not hasattr(model, 'temporal_predictor'):
        return
    
    model.eval()
    data = graphs[year].to(device)
    
    try:
        with torch.no_grad():
            z, _, _ = model.encode(data.x, data.edge_index)
            z_corps = z[:num_corps].cpu().numpy()
            z_patents = z[num_corps:].cpu().numpy()
        
        # グリッド作成
        x_min, x_max = z_corps[:, 0].min(), z_corps[:, 0].max()
        y_min, y_max = z_corps[:, 1].min(), z_corps[:, 1].max()
        margin = 0.5
        
        x = np.linspace(x_min - margin, x_max + margin, 50)
        y = np.linspace(y_min - margin, y_max + margin, 50)
        X, Y = np.meshgrid(x, y)
        
        # ポテンシャル計算用のグリッド入力
        grid_np = np.stack([X.flatten(), Y.flatten()], axis=1)
        grid_tensor = torch.tensor(grid_np, dtype=torch.float32).to(device)
        
        # 時刻 t=0 でのポテンシャル
        t_vec = torch.zeros(grid_tensor.size(0), 1).to(device)
        z_with_t = torch.cat([grid_tensor, t_vec], dim=-1)
        
        with torch.no_grad():
            phi = model.temporal_predictor.potential_net(z_with_t)
            
        Z = phi.cpu().numpy().reshape(X.shape)
        
        fig, ax = plt.subplots(figsize=(10, 8))
        contour = ax.contourf(X, Y, Z, levels=20, cmap='RdYlBu', alpha=0.6)
        plt.colorbar(contour, label='Potential Energy')
        
        # 企業の勾配（移動方向）計算
        z_corps_tensor = torch.tensor(z_corps, dtype=torch.float32, device=device, requires_grad=True)
        t_vec_c = torch.zeros(z_corps_tensor.size(0), 1).to(device)
        
        # 勾配計算のために一時的にenable_grad
        with torch.set_grad_enabled(True):
            z_in_c = torch.cat([z_corps_tensor, t_vec_c], dim=-1)
            phi_c = model.temporal_predictor.potential_net(z_in_c)
            grad = torch.autograd.grad(phi_c.sum(), z_corps_tensor)[0]
        
        grad_np = grad.detach().cpu().numpy()
        
        # プロット
        ax.scatter(z_patents[:, 0], z_patents[:, 1], c='gray', s=10, alpha=0.2, label='Patents')
        ax.scatter(z_corps[:, 0], z_corps[:, 1], c='red', s=50, edgecolors='black', label='Corporations')
        
        # クイバー（ベクトル場）- 勾配の逆方向が力の働く方向
        ax.quiver(z_corps[:, 0], z_corps[:, 1], -grad_np[:, 0], -grad_np[:, 1],
                 color='black', alpha=0.8, width=0.003, scale=20)
        
        ax.set_title(f'Hamiltonian Potential Field (Year {year})')
        ax.legend()
        plt.show()
        
    except Exception as e:
        print(f"可視化エラー: {e}")

# ============================================================
# Main Execution
# ============================================================
if __name__ == "__main__":
    print("=" * 60)
    print("Comparative Experiment: Hamiltonian-ODE vs Baselines")
    print("=" * 60)
    
    # データパスは適宜変更してください
    file_path = 'topic_info3.csv' 
    
    print("\n[Step 1] Data preprocessing...")
    df = preprocess_data(file_path)
    
    if len(df) == 0:
        print("終了: データがありません。")
        exit()
    
    print("\n[Step 2] Building dynamic graphs...")
    (graphs, corps, patents, p_map, total_n, c_map, 
     historical_edges, input_dim) = build_global_graphs(df)
    
    if not graphs:
        print("終了: グラフ構築に失敗しました。")
        exit()

    num_corps = len(corps)
    print(f"✓ Total nodes: {total_n}")
    print(f"✓ Corporations: {num_corps}")
    print(f"✓ Years: {len(graphs)}")
    
    # 実験設定
    models_config = {
        "Static-VGAE": StaticVGAE,
        "LSTM-VGAE": LSTMVGAE,
        "Hamiltonian-ODE": UnifiedVGAE
    }
    
    results = {}
    trained_models = {}
    
    for name, model_cls in models_config.items():
        print(f"\n>>> [Training] {name}")
        
        model = model_cls(
            num_nodes=total_n, 
            num_corps=num_corps, 
            input_dim=input_dim, 
            hidden_dim=64, 
            latent_dim=2 # 可視化用に2次元
        ).to(device)
        
        model, _, history = train_model(
            model, graphs, num_corps, historical_edges, num_epochs=15 # デモ用に短縮
        )
        
        # 最終年の次を予測評価
        test_years = sorted(graphs.keys())[-2:]
        if len(test_years) == 2:
            metrics = compute_ranking_metrics(
                model, 
                graphs[test_years[0]].to(device), 
                graphs[test_years[1]].to(device), 
                num_corps, 
                historical_edges
            )
            results[name] = metrics
            trained_models[name] = model

    # 結果表示
    print("\n" + "="*60)
    print(f"{'Model':<20} | {'MRR':<10} | {'Hits@10':<10}")
    print("-" * 60)
    for name, m in results.items():
        if m:
            print(f"{name:<20} | {m['mrr']:.4f}     | {m['hits@10']:.4f}")
        else:
            print(f"{name:<20} | Failed")
    print("="*60)
    
    # Hamiltonian-ODEの可視化
    if "Hamiltonian-ODE" in trained_models:
        print("\nVisualizing Potential Field...")
        last_year = max(graphs.keys())
        visualize_potential_field(trained_models["Hamiltonian-ODE"], graphs, last_year, device, num_corps)

/home/nakamuraroi/.local/lib/python3.8/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: /home/nakamuraroi/.local/lib/python3.8/site-packages/torch_scatter/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/home/nakamuraroi/.local/lib/python3.8/site-packages/torch_geometric/typing.py:124: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: /home/nakamuraroi/.local/lib/python3.8/site-packages/torch_sparse/_version_cuda.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev
  warnings.warn(f"An issue occurred while importing 'torch-sparse'. "
/home/nakamuraroi/.local/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook imp

Comparative Experiment: Hamiltonian-ODE vs Baselines

[Step 1] Data preprocessing...
1. データ読み込み開始...
エラー: ファイル 'topic_info3.csv' が見つかりません。パスを確認してください。
終了: データがありません。

[Step 2] Building dynamic graphs...


KeyError: 'corporation'

: 